# 01: SMILES Strings - Writing Molecules as Text

## What is this notebook about?

Before a computer can "understand" a molecule, we need a way to write the molecule down as text. Think of it like sheet music: a real symphony is sound, but we can encode it as symbols on a page so it can be stored, shared, and processed.

**SMILES** (Simplified Molecular Input Line Entry System) is the most widely used system for writing molecules as text strings. This notebook explains:
- What SMILES strings are and how to read them
- How to turn SMILES into machine-readable tokens (like words in a sentence)
- Why this matters for machine learning on molecules

---

## The core idea: molecules as sentences

In natural language processing (NLP), a sentence like `"the cat sat"` gets broken into words (tokens): `["the", "cat", "sat"]`. Each word maps to a number. A model processes those numbers.

We do the same with molecules:
- The **molecule** is like a sentence
- The **SMILES string** is how we write that sentence
- **Tokenization** breaks it into atoms and bonds (the "words")
- **Encoding** converts those tokens into numbers a model can process

---

## Part 1: Reading SMILES - the basics

### Atoms
Most atoms are written as their chemical symbol:
- `C` = carbon
- `N` = nitrogen  
- `O` = oxygen
- `F`, `Cl`, `Br`, `I` = halogens (note: `Cl` and `Br` are two-letter tokens!)
- Lowercase letters (`c`, `n`, `o`) = **aromatic** atoms (part of a ring like benzene)

### Bonds
- Single bond: implied (no symbol needed between adjacent atoms)
- Double bond: `=`
- Triple bond: `#`
- Aromatic bond: `:` (or implied between lowercase atoms)

### Branches
- Parentheses `(...)` indicate a branch off the main chain, just like a side chain on an amino acid

### Rings
- Numbers indicate where a ring opens and closes: `C1CCCCC1` = cyclohexane (ring opens at first `1`, closes at second `1`)

### Examples to build intuition

| Molecule | SMILES | What to notice |
|---|---|---|
| Methane | `C` | Just one carbon |
| Ethanol | `CCO` | Two carbons, one oxygen |
| Acetic acid | `CC(=O)O` | Branch: `=O` (double bond oxygen) off the chain |
| Benzene | `c1ccccc1` | Aromatic ring: lowercase + ring closure |
| Aspirin | `CC(=O)Oc1ccccc1C(=O)O` | Ester linkage to an aromatic ring with a carboxylic acid |
| Caffeine | `CN1C=NC2=C1C(=O)N(C(=O)N2C)C` | Two fused rings, multiple nitrogens |

In [1]:
# Let's start by just working with SMILES as plain Python strings.
# No special libraries needed yet!

# A small collection of biologically relevant molecules
molecules = {
    "Aspirin":      "CC(=O)Oc1ccccc1C(=O)O",
    "Ibuprofen":    "CC(C)Cc1ccc(cc1)C(C)C(=O)O",
    "Caffeine":     "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",
    "Paracetamol":  "CC(=O)Nc1ccc(O)cc1",
    "Penicillin G": "CC1([C@@H](N2[C@H](S1)[C@@H](C2=O)NC(=O)Cc3ccccc3)C(=O)O)C",
    "Glucose":      "OC[C@H]1OC(O)[C@H](O)[C@@H](O)[C@@H]1O",
    "Dopamine":     "NCCc1ccc(O)c(O)c1",
    "Serotonin":    "NCCc1c[nH]c2ccc(O)cc12",
}

for name, smiles in molecules.items():
    print(f"{name:15s} | length: {len(smiles):3d} chars | {smiles}")

Aspirin         | length:  21 chars | CC(=O)Oc1ccccc1C(=O)O
Ibuprofen       | length:  26 chars | CC(C)Cc1ccc(cc1)C(C)C(=O)O
Caffeine        | length:  28 chars | CN1C=NC2=C1C(=O)N(C(=O)N2C)C
Paracetamol     | length:  18 chars | CC(=O)Nc1ccc(O)cc1
Penicillin G    | length:  58 chars | CC1([C@@H](N2[C@H](S1)[C@@H](C2=O)NC(=O)Cc3ccccc3)C(=O)O)C
Glucose         | length:  38 chars | OC[C@H]1OC(O)[C@H](O)[C@@H](O)[C@@H]1O
Dopamine        | length:  17 chars | NCCc1ccc(O)c(O)c1
Serotonin       | length:  22 chars | NCCc1c[nH]c2ccc(O)cc12


## Part 2: Tokenization - breaking SMILES into "words"

The first challege is SMILES uses multi-character tokens. `Cl` is one atom (chlorine), not carbon + lowercase-L. Our tokenizer must handle this. 

This is analogous to how in English, `"th"` in "the" is not `t`+`h` separately - some combinations have their own meaning. 

In [3]:
# First, import regex
import re 

# Multi-character tokens must not be split:
# Cl, Br -- two-letter aroms
# @@ -- stereochemistry 
# %nn -- ring closures above ring #9 (e.g., %10, %23)

MULTI_CHAR = re.compile(r'Cl|Br|@@|%\d{2}')

def tokenize(smiles: str) -> list:
    """
    Split a SMILES string into a list of tokens.
    Handles two-character atoms (Cl, Br) and stereochemistry markers.
    """
    tokens = []
    i = 0
    while i < len(smiles):
        match = MULTI_CHAR.match(smiles, i)  # try to match a multi-char token first
        if match:
            tokens.append(match.group())
            i = match.end()
        else:
            tokens.append(smiles[i])         # otherwise, take one character
            i += 1
    return tokens


# Test on a few molecules
test_cases = [
    ("Ethanol",    "CCO"),
    ("Benzene",    "c1ccccc1"),
    ("Aspirin",    "CC(=O)Oc1ccccc1C(=O)O"),
    ("Penicillin", "CC1([C@@H](N2[C@H](S1)[C@@H](C2=O)NC(=O)Cc3ccccc3)C(=O)O)C"),
]

for name, smi in test_cases:
    toks = tokenize(smi)
    print(f"\n{name}")
    print(f"  SMILES: {smi}")
    print(f"  Tokens: {toks}")
    print(f"  Count:  {len(toks)} tokens")


Ethanol
  SMILES: CCO
  Tokens: ['C', 'C', 'O']
  Count:  3 tokens

Benzene
  SMILES: c1ccccc1
  Tokens: ['c', '1', 'c', 'c', 'c', 'c', 'c', '1']
  Count:  8 tokens

Aspirin
  SMILES: CC(=O)Oc1ccccc1C(=O)O
  Tokens: ['C', 'C', '(', '=', 'O', ')', 'O', 'c', '1', 'c', 'c', 'c', 'c', 'c', '1', 'C', '(', '=', 'O', ')', 'O']
  Count:  21 tokens

Penicillin
  SMILES: CC1([C@@H](N2[C@H](S1)[C@@H](C2=O)NC(=O)Cc3ccccc3)C(=O)O)C
  Tokens: ['C', 'C', '1', '(', '[', 'C', '@@', 'H', ']', '(', 'N', '2', '[', 'C', '@', 'H', ']', '(', 'S', '1', ')', '[', 'C', '@@', 'H', ']', '(', 'C', '2', '=', 'O', ')', 'N', 'C', '(', '=', 'O', ')', 'C', 'c', '3', 'c', 'c', 'c', 'c', 'c', '3', ')', 'C', '(', '=', 'O', ')', 'O', ')', 'C']
  Count:  56 tokens
